# 01cities: the 32 cities and their stations

**What it does:** builds the table of the 32 cities in the project. Each city owns one or more stations from Renfe's timetable. The travel time to a city will be the earliest arrival at any of its stations; the dot on the maps sits on its main station. The notebook checks every station against the timetable, measures how far apart a city's stations are, counts how many trips stop there, and draws a quick check map.

**Input:** `data/raw/renfe/<download date>/stops.txt` and `stop_times.txt` (from `00renfegtfs`).

**Output:** `data/clean/cities.csv` and the check map `figures/01cities.png`.

**Data:** Renfe Viajeros, "Horarios de alta velocidad, larga distancia y media distancia", Renfe open data portal (data.renfe.com), under Renfe's open data terms.

## 0. Settings

In [1]:
from pathlib import Path
import math
import pandas as pd
import matplotlib.pyplot as plt

GTFS_FOLDER = Path("data") / "raw" / "renfe" / "2026-11-16"  # <- folder made by 00renfegtfs
OUT_CSV = Path("data") / "clean" / "cities.csv"  # <- the city table
OUT_FIG = Path("figures") / "01cities.png"  # <- the check map
GROUP_COLOURS = {"hub": "#222222", "hsr": "#1D9E75", "left": "#D85A30"}  # <- VISUAL: dot colours per group

# One line per city: name, group, main station id (the dot), other station ids that also count ("" if none)
CITIES = []
CITIES.append(("Madrid", "hub", "60000", "17000"))  # <- Puerta de Atocha; Chamartin
CITIES.append(("Sevilla", "hsr", "51003", ""))  # <- Santa Justa
CITIES.append(("Córdoba", "hsr", "50500", ""))  # <- Julio Anguita
CITIES.append(("Ciudad Real", "hsr", "37200", ""))
CITIES.append(("Cádiz", "hsr", "51405", ""))
CITIES.append(("Málaga", "hsr", "54413", ""))  # <- Maria Zambrano
CITIES.append(("Granada", "hsr", "05000", ""))
CITIES.append(("Zaragoza", "hsr", "04040", ""))  # <- Delicias
CITIES.append(("Lleida", "hsr", "78400", ""))  # <- Pirineus
CITIES.append(("Tarragona", "hsr", "04104", "71500"))  # <- Camp de Tarragona; Tarragona city
CITIES.append(("Barcelona", "hsr", "71801", ""))  # <- Sants
CITIES.append(("Girona", "hsr", "79300", ""))
CITIES.append(("Valencia", "hsr", "03216", "65000"))  # <- Joaquin Sorolla; Estacio del Nord
CITIES.append(("Albacete", "hsr", "60600", ""))  # <- Los Llanos
CITIES.append(("Alicante", "hsr", "60911", ""))  # <- Terminal
CITIES.append(("Murcia", "hsr", "61200", ""))  # <- del Carmen
CITIES.append(("Valladolid", "hsr", "10600", ""))  # <- Campo Grande
CITIES.append(("León", "hsr", "15100", ""))
CITIES.append(("Oviedo", "hsr", "15211", ""))
CITIES.append(("Ourense", "hsr", "22100", ""))
CITIES.append(("Santiago", "hsr", "31400", ""))  # <- Daniel Castelao
CITIES.append(("A Coruña", "hsr", "31412", ""))
CITIES.append(("Vigo", "hsr", "08223", "22308"))  # <- Urzaiz; Guixar
CITIES.append(("Badajoz", "left", "37606", ""))
CITIES.append(("Teruel", "left", "67200", ""))
CITIES.append(("Almería", "left", "56312", ""))
CITIES.append(("Jaén", "left", "03100", ""))
CITIES.append(("Huelva", "left", "43019", ""))
CITIES.append(("Santander", "left", "14223", ""))
CITIES.append(("Bilbao", "left", "13200", ""))  # <- Abando
CITIES.append(("San Sebastián", "left", "11511", ""))
CITIES.append(("Pamplona", "left", "80100", ""))

## 1. Read the timetable stations

In [2]:
def read_gtfs(name):
    table = pd.read_csv(GTFS_FOLDER / name, dtype=str, keep_default_na=False, encoding="utf-8-sig", skipinitialspace=True)
    table.columns = [c.strip() for c in table.columns]
    for column in table.columns:
        table[column] = table[column].str.strip()
    return table

stops = read_gtfs("stops.txt")
stop_times = read_gtfs("stop_times.txt")
stops = stops.set_index("stop_id")
trips_per_stop = stop_times.groupby("stop_id")["trip_id"].nunique()
print(len(stops), "stations,", stop_times["trip_id"].nunique(), "trips")

1006 stations, 6402 trips


## 2. Build the city table and check every station

In [3]:
def km_between(lat1, lon1, lat2, lon2):
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = p2 - p1
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6371 * math.asin(math.sqrt(a))

rows = []
problems = []
for name, group, main_id, other_ids in CITIES:
    ids = [main_id] + [s for s in other_ids.split(";") if s != ""]
    missing = [s for s in ids if s not in stops.index]
    if missing:
        problems.append(f"{name}: station id(s) not in the timetable: {missing}")
        continue
    lat = float(stops.loc[main_id, "stop_lat"])
    lon = float(stops.loc[main_id, "stop_lon"])
    spread = 0.0
    for s in ids[1:]:
        spread = max(spread, km_between(lat, lon, float(stops.loc[s, "stop_lat"]), float(stops.loc[s, "stop_lon"])))
    trips = sum(int(trips_per_stop.get(s, 0)) for s in ids)
    rows.append({"city": name, "group": group, "main_stop_id": main_id, "main_stop_name": stops.loc[main_id, "stop_name"], "stop_ids": ";".join(ids), "stop_names": ";".join(stops.loc[s, "stop_name"] for s in ids), "lat": round(lat, 6), "lon": round(lon, 6), "spread_km": round(spread, 1), "trips_in_feed": trips})

cities = pd.DataFrame(rows)
print("Problems:", problems if problems else "none")
print(len(cities), "cities;", cities["group"].value_counts().to_dict())
print(cities[["city", "group", "stop_names", "spread_km", "trips_in_feed"]].to_string(index=False))

Problems: none
32 cities; {'hsr': 22, 'left': 9, 'hub': 1}
         city group                                                                stop_names  spread_km  trips_in_feed
       Madrid   hub Madrid-Puerta de Atocha-Almudena Grandes;Madrid-Chamartín-Clara Campoamor        7.2           2656
      Sevilla   hsr                                                       Sevilla-Santa Justa        0.0            511
      Córdoba   hsr                                                     Córdoba-Julio Anguita        0.0            653
  Ciudad Real   hsr                                                               Ciudad Real        0.0            336
        Cádiz   hsr                                                                     Cádiz        0.0            191
       Málaga   hsr                                                     Málaga María Zambrano        0.0            172
      Granada   hsr                                                                   Granada        

**Checks to read above:** no problems; 32 cities (1 hub, 22 high-speed, 9 left behind); `spread_km` is the distance between a city's stations (Camp de Tarragona lies outside the city, so a few km is expected); every city should have trips in the timetable.

## 3. Save the table

In [4]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
cities.to_csv(OUT_CSV, index=False, encoding="utf-8")
print("Saved", OUT_CSV, "with", len(cities), "rows")

Saved data\clean\cities.csv with 32 rows


## 4. Check map (draft)

In [5]:
mean_lat = math.radians(cities["lat"].mean())
fig, ax = plt.subplots(figsize=(9, 7.5))
for group, colour in GROUP_COLOURS.items():
    part = cities[cities["group"] == group]
    ax.scatter(part["lon"], part["lat"], s=60 if group == "hub" else 30, color=colour, label=group, zorder=3)
for _, row in cities.iterrows():
    ax.annotate(row["city"], (row["lon"], row["lat"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
ax.set_aspect(1 / math.cos(mean_lat))
ax.set_title("Cities: main station of each city (draft check)")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.grid(color="#e1e0d9", linewidth=0.6)
ax.legend(loc="lower right", frameon=False)
OUT_FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_FIG, dpi=150, bbox_inches="tight")
plt.show()
print("Saved", OUT_FIG)

Saved figures\01cities.png


___
**END**